본 실습 노트북은 **모두의 AI** - 실전! RAG 고급 기법 1, 2 강의를 참고하여 작성되었습니다.
LangChain v1.x 업데이트로 인해 deprecated된 라이브러리들을 최신 패키지로 전면 교체하였습니다.

In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -q langchain-text-splitters langchain-chroma langchain-huggingface

In [ ]:
!pip install -q langchain-community beautifulsoup4

In [ ]:
!pip install -q langchain-classic==0.0.2 langchain-community==0.3.0

ERROR: Could not find a version that satisfies the requirement langchain-classic==0.0.2 (from versions: 1.0.0a1, 1.0.0, 1.0.1, 1.0.2, 1.0.3, 1.0.4, 1.0.5, 1.0.6, 1.0.7)
ERROR: No matching distribution found for langchain-classic==0.0.2


In [ ]:
import os
from google.colab import userdata

# ⚠️ API 키 설정 - 아래 두 방법 중 하나를 선택하세요

# 방법 1: Colab Secrets 사용 (권장)
# 좌측 사이드바 🔑 아이콘 → OPENAI_API_KEY 등록 후 아래 주석 해제
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# 방법 2: 직접 입력 (노트북 공유 시 키 노출 위험)
os.environ['OPENAI_API_KEY'] = 'YOUR_API_KEY_HERE'


## Multi-Query Retriever

### 개념
Multi-Query Retriever는 하나의 질문을 **여러 개의 다양한 질문으로 자동 변환**하여 검색하는 기법입니다.

### 동작 방식
1. 사용자의 원본 질문을 LLM이 관점이 다른 여러 질문으로 재작성
2. 각 질문으로 벡터DB를 개별 검색
3. 모든 결과를 합친 후 중복 제거

### 장점
- 단일 질문의 표현 한계를 극복
- 다양한 관점의 문서를 검색하여 recall 향상
- 사용자가 질문을 정확히 표현하지 못해도 관련 문서 검색 가능


In [ ]:
# Build a sample vectorDB
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Load blog post
loader = WebBaseLoader("https://n.news.naver.com/mnews/article/003/0012317114?sid=105")
data = loader.load()

# # Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# VectorDB
model_name = "jhgan/ko-sbert-nli"
encode_kwargs = {'normalize_embeddings': True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

vectordb = Chroma.from_documents(documents=splits, embedding=ko_embedding)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jhgan/ko-sbert-nli
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install -q langchain-openai

In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI
import os

question = "삼성전자 갤럭시 S24는 어떨 예정이야?"
llm = ChatOpenAI(temperature=0, openai_api_key=os.environ.get('OPENAI_API_KEY'))
retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm
)


In [ ]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [ ]:
unique_docs = retriever_from_llm.invoke(question)
len(unique_docs)

8

In [ ]:
unique_docs

[Document(id='7a055b91-86ff-4368-940d-b7bc97d173c4', metadata={'language': 'ko', 'title': "언팩 D-4, 세계 최초 AI폰 '갤S24' 이렇게 나온다", 'source': 'https://n.news.naver.com/mnews/article/003/0012317114?sid=105'}, page_content="[서울=뉴시스] 삼성전자가 17일 오전 10시(현지시간, 한국 시간 18일 오전 3시) 미국 캘리포니아주 산호세(새너제이)에서 '삼성 갤럭시 언팩 2024'를 열고 갤럭시 S24를 공개한다. 사진은 포르투갈에서 유출된 갤럭시 S24 시리즈 포스터 추정 이미지 (사진=theonecid 엑스 캡처)  *재판매 및 DB 금지[서울=뉴시스]윤정민 기자 = 인공지능(AI) 서비스가 대거 탑재될 삼성전자 플래그십 스마트폰 '갤럭시 S24'가 18일 베일을 벗는다. 갤럭시 S23이 전작 대비 카메라, 디자인 등 대폭 개선됐다면, 이번 신작은"),
 Document(id='cb2ff053-6759-4b72-8fde-2e3b59d6b0a8', metadata={'language': 'ko', 'title': "언팩 D-4, 세계 최초 AI폰 '갤S24' 이렇게 나온다", 'source': 'https://n.news.naver.com/mnews/article/003/0012317114?sid=105'}, page_content="[서울=뉴시스] 삼성전자가 17일 오전 10시(현지시간, 한국 시간 18일 오전 3시) 미국 캘리포니아주 산호세(새너제이)에서 '삼성 갤럭시 언팩 2024'를 열고 갤럭시 S24를 공개한다. 사진은 포르투갈에서 유출된 갤럭시 S24 시리즈 포스터 추정 이미지 (사진=theonecid 엑스 캡처)  *재판매 및 DB 금지[서울=뉴시스]윤정민 기자 = 인공지능(AI) 서비스가 대거 탑재될 삼성전자 플래그십 스마트폰 '갤럭시 S24'가 18일 베일을 벗는다. 갤럭시 S23이 전

## 기본 Parent-document Retriever

### 개념
Parent-Document Retriever는 **작은 청크(child)**로 검색하고, **큰 청크(parent)**를 반환하는 기법입니다.

### 동작 방식
1. 문서를 큰 단위(parent)와 작은 단위(child)로 분할
2. 벡터DB에는 작은 청크(child)를 인덱싱
3. 검색 시 작은 청크로 유사도 검색
4. 실제 반환은 해당 child가 속한 parent 문서 전체를 반환

### 장점
- 검색 정확도(precision)와 문맥 풍부함(context) 동시 확보
- 작은 청크로 정밀 검색 → 큰 청크로 충분한 문맥 제공


In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever

In [ ]:
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
loaders = [
    PyPDFLoader("/content/drive/MyDrive/[복지이슈 FOCUS 15ȣ] 경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색.pdf"),
    PyPDFLoader("/content/drive/MyDrive/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load_and_split())

In [ ]:
model_name = "jhgan/ko-sbert-nli"
encode_kwargs = {'normalize_embeddings': True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jhgan/ko-sbert-nli
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# This text splitter is used to create the child documents
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="full_documents", embedding_function=ko_embedding
)
# The storage layer for the parent documents
store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

In [ ]:
retriever.add_documents(docs, ids=None)

In [ ]:
sub_docs = vectorstore.similarity_search("인공지능 예산")

In [ ]:
print("글 길이: {}\n\n".format(len(sub_docs[0].page_content)))
print(sub_docs[0].page_content)

글 길이: 364


하드웨어30.943.160.289.2118.3176.541.7%　(39.5)(39.7)(48.2)(32.5)(49.2)       자료: Global Artificial Intelligence(AI) Market, BCC Research (2022)￮인공지능 기술을 도입하는 산업이 늘어나고 있는 점, 인공지능 분야에 진출하는 스타트업의 증가에 따라 산업의 기술경쟁력이 높아지는 점 등은 시장성장에 촉진요인으로 작용할 전망임￮국내 인공지능 기술에 대한 완성도가 높지 않아 국내기술의 도입처가 제한적인 점, 인공지능이 인간의 고용 영역을 침범할 수 있고 기술의 불안정성으로 인한 사고 발생의 우려가 존재하는 점 등은 시장성장에 저해요인으로 작용할 전망임


In [ ]:
retrieved_docs = retriever.invoke("인공지능 예산")

In [ ]:
print("글 길이: {}\n\n".format(len(retrieved_docs[0].page_content)))
print(retrieved_docs[0].page_content)

글 길이: 1261


| 10 | CIS이슈리포트 2022-2호 
▶인공지능 산업의 value chain은 ‘AI 플랫폼 공급업체 → AI 어플리케이션 개발 → AI 응용솔루션 개발 → 이용자’로 구성되며, 동 산업은 ①성장기 산업, ②대체재로부터의 위협이 낮은 산업, ③기술집약적 산업 등의 특징을 가짐￮알고리즘, 하드웨어 기술개발과 응용솔루션 서비스 상용화가 활발히 진행 중인 성장기 산업이며, 수요 기업의 요구사항에 따라 운영플랫폼을 선택할 수 있는 구매자의 교섭력이 높은 산업임￮직접적인 대체 기술이 없어 대체재로부터 위협이 낮은 편이며, 알고리즘의 동작원리를 이해하고 맞춤형 서비스를 지원하기 위한 솔루션 개발 능력이 뒷받침 되어야 하는 기술집약적 산업임▶시장조사전문기관 BCC research에 따르면 세계 인공지능 시장규모는 2020년 398.4억 달러에서 연평균 41.0% 성장하여 2025년에는 2,223.7억 달러의 시장을 형성할 것으로 전망됨￮세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 정도로 압도적으로 높음[세계 인공지능 시장규모]                                                            (단위: 억 달러, 괄호는 YoY %)구분202020212022202320242025CAGR(2020-2025)인공지능398.4553.3769.71,134.31,498.92,223.741.0%　(38.9)(39.1)(47.4)(32.1)(48.4)  소프트웨어311.8432.3600.3882.41,164.61,723.540.8%　(38.6)(38.8)(47.0)(32.0)(48.0)  서비스55.778.0109.3162.6216.0323.742.2%　(40.0)(40.2)(48.9)(32.8)(49.8)  하드웨어30.943.160.289.2118.3176.541.7%　(39.5)(39.7)(48.2)(32.5)(49.2)       자료: Global Artifi

## 본문의 Full_chunk가 너무 길때

### 개념
원본 문서 전체가 너무 길 경우, parent도 적당한 크기로 분할하여 관리하는 방식입니다.

### 동작 방식
- `parent_splitter`: 큰 단위로 문서 분할 (예: chunk_size=800)
- `child_splitter`: parent보다 작은 단위로 분할 (예: chunk_size=200)
- 검색은 child로, 반환은 parent로

### 장점
- 원본 문서가 매우 길어도 효율적으로 처리 가능
- LLM의 컨텍스트 한계 내에서 최대한 풍부한 문맥 제공


In [ ]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=ko_embedding
)
# The storage layer for the parent documents
store = InMemoryStore()

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [ ]:
retriever.add_documents(docs)

In [ ]:
len(list(store.yield_keys()))

86

In [ ]:
sub_docs = vectorstore.similarity_search("인공지능 예산")

In [ ]:
print(sub_docs[0].page_content)

내 인공지능 품목이 정책금융 공급량 및 공급속도 증가 측면에서 정책금융의 흐름을 이끌고 있음￮초연결 사회 구축을 위한 차세대 이동통신 시스템 기술의 발전, 빅데이터 및 컴퓨팅 기술의 발전에 따른 인공지능의 상용화 영향 때문인 것으로 분석됨▶전기전자 테마의 경우 차세대반도체 기술분야 내 시스템반도체 품목이 미래성장성에 기반하여 큰 규모의 정책자금 투입을


In [ ]:
len(sub_docs[0].page_content)

197

In [ ]:
retrieved_docs = retriever.invoke("인공지능 예산")

In [ ]:
print(retrieved_docs[0].page_content)

6. 요약 및 결언▶혁신성장 ICT 산업에 지원된 정책금융은 16.9조원(21년 말 기준) 규모로, 전체 혁신성장 정책금융 총량의 약 20% 비중을 차지하고 있으며, 지원규모가 매년 증가하고 있음▶정보통신 테마의 경우 차세대무선통신미디어 기술분야 내 5G이동통신 품목과 능동형컴퓨팅 기술 분야 내 인공지능 품목이 정책금융 공급량 및 공급속도 증가 측면에서 정책금융의 흐름을 이끌고 있음￮초연결 사회 구축을 위한 차세대 이동통신 시스템 기술의 발전, 빅데이터 및 컴퓨팅 기술의 발전에 따른 인공지능의 상용화 영향 때문인 것으로 분석됨▶전기전자 테마의 경우 차세대반도체 기술분야 내 시스템반도체 품목이 미래성장성에 기반하여 큰 규모의 정책자금 투입을 유발하고 있음￮종합반도체 강국을 목표로 하는 비전과 반도체 특별법 제정 등을 통한 산업 육성화 정책으로 인해 동 품목으로의 정책금융 공급은 지속 증가할 것으로 전망됨▶센서측정 테마의 경우 정보통신, 전기전자 테마 대비 정책금융 공급 규모는 작은 편이나, 객체탐지 분야로의 정책금융 공급이 꾸준한 것을 확인함￮스마트팜, 자율주행차 등 스마트센서를 필요로 하는 산업으로부터의 수요가 증가함에 따라 동 품목 시장의 성장이 전망되며, 이에 정책금융 공급 또한 지속 증가할 것임▶원천기술 경쟁력 강화 등에 혁신성장 정책금융이 중요한 역할을 하고 있으며, 미래먹거리 산업 육성을 위해 역동적인 혁신금융으로서의 변화가 기대됨￮혁신 ICT 산업은 관련 시장이 지속적으로 성장할 것으로 전망되나, 원천기술 미확보 및 높은 해외 의존도가 약점으로 지적되어 국내 기업의 경쟁력 강화가 필요함￮이에


In [ ]:
len(retrieved_docs[0].page_content)

796

## Self-querying

### 개념
Self-Query Retriever는 자연어 질문에서 **의미 검색 쿼리**와 **메타데이터 필터**를 자동으로 분리·생성하는 기법입니다.

### 동작 방식
1. LLM이 자연어 질문을 분석
2. 벡터 검색용 의미 쿼리와 필터 조건(연도, 장르, 평점 등)을 자동 추출
3. 추출된 필터를 메타데이터 조건으로 벡터DB에 적용

### 장점
- "2010년 이후 평점 8점 이상 SF 영화"같은 복합 조건 질문 처리 가능
- 메타데이터 필터링과 의미 검색을 동시에 활용


In [ ]:
!pip install lark

In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]
vectorstore = Chroma.from_documents(docs, ko_embedding)

In [ ]:
!pip install -U langchain-community langchain

In [ ]:
!pip install --upgrade databricks-langchain langchain-community langchain


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_openai import ChatOpenAI
import os

metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]
document_content_description = "Brief summary of a movie"
llm = ChatOpenAI(temperature=0, openai_api_key=os.environ.get('OPENAI_API_KEY'))
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectorstore,
    document_content_description,
    metadata_field_info,
    verbose=True
)


In [ ]:
retriever.invoke("what are some movies rated higher than 8.5")

[Document(id='e411249d-84fc-4108-a0f6-d8d99ae7acb6', metadata={'year': 1979, 'genre': 'thriller', 'rating': 9.9, 'director': 'Andrei Tarkovsky'}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(id='67f8dd34-b9a0-4dbc-8ed0-2778fc4b46b4', metadata={'year': 2006, 'director': 'Satoshi Kon', 'rating': 8.6}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea')]

## Time-weighted vector store Retriever

### 개념
Time-Weighted Retriever는 **의미적 유사도**와 **시간 가중치**를 결합하여 최신 문서를 우선적으로 반환하는 기법입니다.

### 스코어링 공식
`score = semantic_similarity + (1.0 - decay_rate) ^ hours_passed`

### 파라미터
- `decay_rate`가 높을수록 → 최신 문서 우선 (빠르게 과거 문서 점수 감소)
- `decay_rate`가 낮을수록 → 과거 문서도 잘 검색됨

### 장점
- 뉴스, 공지사항 등 최신성이 중요한 도메인에 적합
- 의미 유사도와 시간 요소를 균형있게 반영


Scoring 방법 = *semantic_similarity + (1.0 - decay_rate) ^ hours_passed*

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.0 MB/s eta 0:00:00


In [ ]:
from datetime import datetime, timedelta

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore                                    # 수정
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever         # 수정
from langchain_core.documents import Document                                      # 수정
from langchain_community.vectorstores import FAISS                                 # 유지

In [ ]:
# Initialize the vectorstore as empty
embedding_size = 768
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(ko_embedding, index, InMemoryDocstore({}), {})
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore, decay_rate=0.99, k=1
) #decay_rate = 0.01이면 과거 값을 찾는다.

In [ ]:
yesterday = datetime.now() - timedelta(days=1)
retriever.add_documents(
    [Document(page_content="영어는 훌륭합니다.", metadata={"last_accessed_at": yesterday})]
)
retriever.add_documents([Document(page_content="한국어는 훌륭합니다")])

['fbe643a4-92f4-4448-82d8-52c2aab51d7b']

In [ ]:
# "Hello World" is returned first because it is most salient, and the decay rate is close to 0., meaning it's still recent enough
retriever.invoke("영어가 좋아요")

[Document(metadata={'last_accessed_at': datetime.datetime(2026, 5, 25, 11, 58, 7, 816982), 'created_at': datetime.datetime(2026, 5, 25, 11, 57, 46, 297640), 'buffer_idx': 1}, page_content='한국어는 훌륭합니다')]

## Ensemble Retriever

### 개념
Ensemble Retriever는 **BM25(키워드 기반)**와 **Dense Retrieval(의미 기반)**을 결합하여 검색 성능을 높이는 기법입니다.

### 동작 방식
1. BM25 Retriever: 키워드 매칭 기반 검색 (정확한 단어 포함 문서에 강함)
2. FAISS Retriever: 임베딩 기반 의미 검색 (유사한 의미의 문서에 강함)
3. 두 결과를 가중치(weights)에 따라 합산하여 최종 결과 반환

### 파라미터
- `weights=[0.5, 0.5]`: BM25와 Dense 검색의 가중치 조절
- 키워드 중심 도메인 → BM25 가중치 높임
- 의미 중심 도메인 → Dense 가중치 높임

### 장점
- 키워드 검색과 의미 검색의 단점을 상호 보완
- 다양한 유형의 질문에 robust한 검색 성능


In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-huggingface langchain-classic faiss-cpu rank_bm25 pypdf sentence-transformers chromadb

In [ ]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
model_name = "jhgan/ko-sbert-nli"
encode_kwargs = {'normalize_embeddings': True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jhgan/ko-sbert-nli
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter       # 수정
from langchain_community.document_loaders import PyPDFLoader

loaders = [
    PyPDFLoader("/content/drive/MyDrive/[복지이슈 FOCUS 15ȣ] 경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색.pdf"),
    PyPDFLoader("/content/drive/MyDrive/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf"),
]
docs = []
for loader in loaders:
    docs.extend(loader.load_and_split())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(docs)

In [ ]:
# initialize the bm25 retriever and faiss retriever
bm25_retriever = BM25Retriever.from_documents(texts)
bm25_retriever.k = 2 #BM25 알고리즘을 통해 검색되는 문서 2개

embedding = ko_embedding
faiss_vectorstore = FAISS.from_documents(texts, ko_embedding)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5] #가중치에 따라 키워드 기반일건지, Dense기반일건지 조절
)

In [ ]:
docs = ensemble_retriever.invoke("혁신정책금융과 극저신용대출모형의 차이")
for i in docs:

  print(i.metadata)
  print(":")
  print(i.page_content)
  print("-"*100)

{'producer': 'Hancom PDF 1.3.0.538', 'creator': 'Hancom PDF 1.3.0.538', 'creationdate': '2022-07-29T09:03:16+09:00', 'author': 'kmd kdy', 'moddate': '2022-07-29T09:03:16+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf', 'total_pages': 18, 'page': 17, 'page_label': '18'}
:
※ 본 보고서의 내용은 작성자 개인의 의견으로서 한국신용정보원의 공식 견해와 다를 수 있습니다.     본 보고서를 사용 또는 인용할 경우에는 출처를 명시하시기 바랍니다.
----------------------------------------------------------------------------------------------------
{'producer': 'Hancom PDF 1.3.0.509', 'creator': 'Hancom PDF 1.3.0.509', 'creationdate': '2021-12-28T13:18:22+09:00', 'author': 'User', 'moddate': '2021-12-28T13:18:22+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/[복지이슈 FOCUS 15ȣ] 경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색.pdf', 'total_pages': 20, 'page': 6, 'page_label': '7'}
:
◯도내 저신용･저소득층의 극저신용대출 수요증대에 따라 대출신청 시 금융 및 비금융 정보수집에 기반한 변

In [ ]:
faiss_vectorstore = FAISS.from_documents(texts, ko_embedding)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 4})

docs = faiss_retriever.invoke("혁신정책금융과 극저신용대출모형의 차이")
for i in docs:

  print(i.metadata)
  print(":")
  print(i.page_content)
  print("-"*100)

{'producer': 'Hancom PDF 1.3.0.509', 'creator': 'Hancom PDF 1.3.0.509', 'creationdate': '2021-12-28T13:18:22+09:00', 'author': 'User', 'moddate': '2021-12-28T13:18:22+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/[복지이슈 FOCUS 15ȣ] 경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색.pdf', 'total_pages': 20, 'page': 6, 'page_label': '7'}
:
◯도내 저신용･저소득층의 극저신용대출 수요증대에 따라 대출신청 시 금융 및 비금융 정보수집에 기반한 변별력을 갖춘 심사체계 마련이 요구되는 상황-경기극저신용대출은 기존 신용평가체계(10등급 기준)를 활용하여 신용정보회사(NICE, KCB)에서 제공하는 10등급 기준 7등급 이하의 극저신용자를 모집단으로 설정, 극저신용대출 수행기관(사회연대은행, 주빌리은행, 지역자활협회)을 통해 신청 및 심사절차를 거쳐 최종 대상자를 선정함-현행 극저신용대출 심사기준(2020)에 따르면 차주 관련 기초현황점수(정량지표, 55점)와 심사점수(정성지표, 45점)로 구분되어 있고, 심사기준별 세부심사항목은 인구학적 정보(성별, 연령, 거주지, 직업, 가구원, 주거형태), 경제상황(정부지원 여부, 소득유무, 소득유형, 가계소득, 연체유형, 부채규모), 신용정보(신용등급, 채무조정유형), 대출 관련 평가(대출용도, 상환계획, 긴급성, 상환가능성 등) 내용으로 구성됨-1차년도 극저신용대출의 심사기준은
----------------------------------------------------------------------------------------------------
{'pro

In [ ]:
import os
from google.colab import userdata

# Colab Secrets에 OPENAI_API_KEY를 등록한 후 아래 방법 중 하나를 선택하세요
# 방법 1: Colab Secrets 사용 (권장)
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# 방법 2: 직접 입력 (비권장 - 키 노출 주의)
# os.environ['OPENAI_API_KEY'] = 'YOUR_API_KEY_HERE'

from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI

openai = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

qa = RetrievalQA.from_chain_type(llm=openai,
                                 chain_type="stuff",
                                 retriever=ensemble_retriever,
                                 return_source_documents=True)

query = "극저신용자 대출의 신용등급"
result = qa(query)
print(result['result'])


In [ ]:
for i in result['source_documents']:
  print(i.metadata)
  print("-"*100)
  print(i.page_content)
  print("-"*100)

{'producer': 'Hancom PDF 1.3.0.509', 'creator': 'Hancom PDF 1.3.0.509', 'creationdate': '2021-12-28T13:18:22+09:00', 'author': 'User', 'moddate': '2021-12-28T13:18:22+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/[복지이슈 FOCUS 15ȣ] 경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색.pdf', 'total_pages': 20, 'page': 11, 'page_label': '12'}
----------------------------------------------------------------------------------------------------
현장공감 경기복지재단 12
경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색
-2021년 6월말 기준 신용등급 대상자(47,307,806명) 가운데 27.1%(12,807.275명)이 금융이력부족자로 나타났고, 60대 이상(4,179,087명), 20대(3,227,319명), 30대(1,723,466명), 50대(1,450,496명) 순으로 20대 청년과 60대 이상이 절반 이상을 차지함-더구나 대다수 금융이력부족자들은 700점대의 낮은 신용점수(CB등급 기준 4~7등급)를 받는 것으로 나타나 금융이력부족군 중 상당수의 저신용 금융취약계층이 분포되어 있음을 확인-결과적으로 금융거래정보 위주의 개인신용평가모형은 금융소외계층에 대한 금융불평등 문제를 지속적으로 양산할 수밖에 없는 한계를 지니고 있어 이들을 위한 더욱 정교하고 공평한 신용평가모형 개발에 대한 요구 증대
--------------------------------------------------------

In [ ]:
faiss_vectorstore = FAISS.from_documents(docs, ko_embedding)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 4})

qa = RetrievalQA.from_chain_type(llm = openai,
                                 chain_type = "stuff",
                                 retriever = faiss_retriever,
                                 return_source_documents = True)

query = "극저신용자 대출의 신용등급"
result = qa(query)
print(result['result'])

극저신용자 대출의 경우에는 7등급 이하의 극저신용자를 대상으로 하고 있습니다. 이들은 기존의 10등급 기준에서 7등급 이하로 분류되어 대출을 받을 수 있습니다.


In [ ]:
for i in result['source_documents']:
  print(i.metadata)
  print("-"*100)
  print(i.page_content)
  print("-"*100)

{'producer': 'Hancom PDF 1.3.0.509', 'creator': 'Hancom PDF 1.3.0.509', 'creationdate': '2021-12-28T13:18:22+09:00', 'author': 'User', 'moddate': '2021-12-28T13:18:22+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/[복지이슈 FOCUS 15ȣ] 경기도 극저신용대출심사모형 개발을 위한 국내 신용정보 활용가능성 탐색.pdf', 'total_pages': 20, 'page': 6, 'page_label': '7'}
----------------------------------------------------------------------------------------------------
◯도내 저신용･저소득층의 극저신용대출 수요증대에 따라 대출신청 시 금융 및 비금융 정보수집에 기반한 변별력을 갖춘 심사체계 마련이 요구되는 상황-경기극저신용대출은 기존 신용평가체계(10등급 기준)를 활용하여 신용정보회사(NICE, KCB)에서 제공하는 10등급 기준 7등급 이하의 극저신용자를 모집단으로 설정, 극저신용대출 수행기관(사회연대은행, 주빌리은행, 지역자활협회)을 통해 신청 및 심사절차를 거쳐 최종 대상자를 선정함-현행 극저신용대출 심사기준(2020)에 따르면 차주 관련 기초현황점수(정량지표, 55점)와 심사점수(정성지표, 45점)로 구분되어 있고, 심사기준별 세부심사항목은 인구학적 정보(성별, 연령, 거주지, 직업, 가구원, 주거형태), 경제상황(정부지원 여부, 소득유무, 소득유형, 가계소득, 연체유형, 부채규모), 신용정보(신용등급, 채무조정유형), 대출 관련 평가(대출용도, 상환계획, 긴급성, 상환가능성 등) 내용으로 구성됨-1차년도 극저신용대출의 심사기준은
-------

## Long Context Reorder

### 개념
LLM은 긴 문맥에서 **중간에 위치한 정보를 잘 활용하지 못하는 경향(Lost in the Middle)**이 있습니다.
Long Context Reorder는 검색된 문서를 **앞부분과 뒷부분에 중요 문서를 배치**하여 이 문제를 해결합니다.

### 동작 방식
1. 관련성 순으로 정렬된 문서를 받음
2. 가장 관련성 높은 문서를 앞/뒤로 재배치
3. 관련성 낮은 문서를 중간에 배치

### 장점
- LLM의 "Lost in the Middle" 문제 완화
- 동일한 검색 결과로 더 정확한 답변 생성 가능
- 추가 비용 없이 성능 향상


In [ ]:
from langchain_classic.chains import LLMChain, StuffDocumentsChain
from langchain_core.prompts import PromptTemplate
from langchain_community.document_transformers import (
    LongContextReorder,
)
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI

texts = [
    "바스켓볼은 훌륭한 스포츠입니다.",
    "플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.",
    "셀틱스는 제가 가장 좋아하는 팀입니다.",
    "보스턴 셀틱스에 관한 문서입니다.", "보스턴 셀틱스는 제가 가장 좋아하는 팀입니다.",
    "저는 영화 보러 가는 것을 좋아해요",
    "보스턴 셀틱스가 20점차로 이겼어요",
    "이것은 그냥 임의의 텍스트입니다.",
    "엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.",
    "L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.",
    "래리 버드는 상징적 인 NBA 선수였습니다.",
]

# Create a retriever
retriever = Chroma.from_texts(texts, embedding=ko_embedding).as_retriever(
    search_kwargs={"k": 10}
)
query = "셀틱스에 대해 어떤 이야기를 들려주시겠어요?"

# Get relevant documents ordered by relevance score
docs = retriever.invoke(query)
docs

[Document(metadata={}, page_content='보스턴 셀틱스에 관한 문서입니다.'),
 Document(metadata={}, page_content='보스턴 셀틱스에 관한 문서입니다.'),
 Document(metadata={'director': 'Satoshi Kon', 'year': 2006, 'rating': 8.6}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'title': "언팩 D-4, 세계 최초 AI폰 '갤S24' 이렇게 나온다", 'language': 'ko', 'source': 'https://n.news.naver.com/mnews/article/003/0012317114?sid=105'}, page_content='영종 버스 차고지 식당서 60여명 집단 식중독 증세\n\n\n\n\n\n\n\n\n\n\n\n6위\n\n\n\n\n종전 협상 막판 신경전…美 "합의 안되면 다른 방식" vs 이란 "核은 논의 안해"\n\n\n\n\n\n\n\n랭킹 뉴스 더보기\n\n\n\n\n\n\n\n\n\n네이버 AI 뉴스 알고리즘 뉴스 추천 알고리즘이 궁금하다면?\n\n\n\n\n\n\n\n\n\n\n\n이슈 NOW\n\n안내\n\n\n언론사에서 직접 선별한 이슈입니다.\n\n닫기\n\n\n\n\n\n내 구독 이슈\n\n\n\n\n\n\n\n\n트럼프\n\n\n\n\n부동산\n\n\n\n\n김건희\n\n\n\n\n이란\n\n\n\n\n재보궐\n\n\n\n\n인공지능\n\n\n\n\n정책\n\n\n\n\n이재명\n\n\n\n\n서울시장'),
 Document(metadata={'language': 'ko', 'source': 'https://n.news.naver.com/mnews/article/003/001

In [ ]:
reordering = LongContextReorder()
reordered_docs = reordering.transform_documents(docs)

# Confirm that the 4 relevant documents are at beginning and end.
reordered_docs

[Document(metadata={}, page_content='보스턴 셀틱스에 관한 문서입니다.'),
 Document(metadata={'title': "언팩 D-4, 세계 최초 AI폰 '갤S24' 이렇게 나온다", 'language': 'ko', 'source': 'https://n.news.naver.com/mnews/article/003/0012317114?sid=105'}, page_content='영종 버스 차고지 식당서 60여명 집단 식중독 증세\n\n\n\n\n\n\n\n\n\n\n\n6위\n\n\n\n\n종전 협상 막판 신경전…美 "합의 안되면 다른 방식" vs 이란 "核은 논의 안해"\n\n\n\n\n\n\n\n랭킹 뉴스 더보기\n\n\n\n\n\n\n\n\n\n네이버 AI 뉴스 알고리즘 뉴스 추천 알고리즘이 궁금하다면?\n\n\n\n\n\n\n\n\n\n\n\n이슈 NOW\n\n안내\n\n\n언론사에서 직접 선별한 이슈입니다.\n\n닫기\n\n\n\n\n\n내 구독 이슈\n\n\n\n\n\n\n\n\n트럼프\n\n\n\n\n부동산\n\n\n\n\n김건희\n\n\n\n\n이란\n\n\n\n\n재보궐\n\n\n\n\n인공지능\n\n\n\n\n정책\n\n\n\n\n이재명\n\n\n\n\n서울시장'),
 Document(metadata={'source': 'https://n.news.naver.com/mnews/article/003/0012317114?sid=105', 'language': 'ko', 'title': "언팩 D-4, 세계 최초 AI폰 '갤S24' 이렇게 나온다"}, page_content='AI·반도체 패권경쟁\n\n\n\n구독\n\n\n구독중\n\n\n\n\n삼성전자 잠정합의안 투표율 80%⋯일부 주주 "합의 무효"\n\n\n\n\n\n\n\n\n\n\n\n\n\n인공지능전환(AX) 시대\n\n\n\n구독\n\n\n구독중\n\n\n\n\n\'완성차 중 피지컬AI 가장 빠르다\'...NH, 현대차 목표주가 86만원 제시\n\n\

In [ ]:
from langchain_classic.chains import LLMChain, StuffDocumentsChain
from langchain_core.prompts import PromptTemplate
import os

from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI

document_prompt = PromptTemplate(
    input_variables=["page_content"], template="{page_content}"
)

template = """Given this text extracts:
-----
{context}
-----
Please answer the following question:
{query}"""
prompt = PromptTemplate(
    template=template, input_variables=["context", "query"]
)
openai = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

llm_chain = LLMChain(llm=openai, prompt=prompt)
chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_prompt=document_prompt,
    document_variable_name="context"
)


In [ ]:
reordered_result = chain.run(input_documents=reordered_docs, query=query)
result = chain.run(input_documents=docs, query=query)

print(reordered_result)
print("-"*100)
print(result)

/tmp/ipykernel_41102/2361813174.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  reordered_result = chain.run(input_documents=reordered_docs, query=query)


셀틱스는 제가 가장 좋아하는 팀이며, L. 코넷은 최고의 셀틱스 선수 중 한 명이라고 언급되었습니다.
----------------------------------------------------------------------------------------------------
셀틱스는 제가 가장 좋아하는 팀이며, L. 코넷은 최고의 셀틱스 선수 중 한 명이라고 합니다.
